# 02. Subset 선정 검증

공식 Train / Validation / Test split은 그대로 유지하고,
각 split에서 약 1/3만 추출하여 실험용 subset을 구성.

예상 크기
- Train: 5200 → 1733
- Val: 2600 → 867
- Test: 2200 → 733


In [5]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# 프로젝트 경로
ROOT = Path("..")
SOURCE_ROOT = ROOT / "data" / "yolo"

# 각 공식 split에서 1/3만 사용
SUBSET_RATIO = 1 / 3

SPLITS = ("train", "val", "test")
SEED = 42
AREA_BINS = 5
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

print(SOURCE_ROOT.resolve())


C:\Users\PMS\Desktop\MS\project\small-drone-detection\data\yolo


## 1. 이미지별 bbox 면적과 객체 수 정리

YOLO 라벨의 `width × height`로 bbox 면적 비율 계산.
이미지마다 bbox 면적 목록과 객체 수를 저장.


In [6]:
def load_records(split):
    """이미지별 bbox 면적과 객체 수 정리."""
    image_dir = SOURCE_ROOT / "images" / split
    label_dir = SOURCE_ROOT / "labels" / split
    records = []

    for image in sorted(image_dir.iterdir()):
        if image.suffix.lower() not in IMAGE_EXTS:
            continue

        label = label_dir / f"{image.stem}.txt"
        areas = []

        # YOLO: class x_center y_center width height
        if label.exists():
            for line in label.read_text(encoding="utf-8").splitlines():
                p = line.split()
                if len(p) >= 5:
                    areas.append(float(p[3]) * float(p[4]))

        records.append({
            "image": image,
            "label": label,
            "areas": areas,
            "count": len(areas),
        })

    return records


## 2. Subset 선정

- 단일 객체: bbox가 작은 순으로 정렬 → 5등분 → 각 구간에서 골고루 추출
- Train: 배경·다중 객체는 모두 포함
- Val/Test: 배경·단일·다중 객체 비율을 원본과 비슷하게 유지


In [7]:
def sample_single(records, target_size, rng):
    """단일 객체를 bbox 크기별로 골고루 추출."""
    if target_size >= len(records):
        return list(records)

    # bbox 작은 순 → 큰 순
    records = sorted(records, key=lambda r: r["areas"][0])

    # 5개 크기 구간
    groups = np.array_split(records, AREA_BINS)

    # 각 구간에서 비슷한 수량 추출
    base, extra = divmod(target_size, AREA_BINS)
    selected = []

    for i, group in enumerate(groups):
        take = base + (1 if i < extra else 0)
        selected += rng.sample(list(group), take)

    return selected


def select_subset(records, split, target_size, rng):
    """Train / Val / Test 목적에 맞게 subset 선정."""
    empty = [r for r in records if r["count"] == 0]
    single = [r for r in records if r["count"] == 1]
    multi = [r for r in records if r["count"] >= 2]

    if split == "train":
        # 희소 사례는 학습에 전부 사용
        fixed = empty + multi
        selected = fixed + sample_single(
            single,
            target_size - len(fixed),
            rng,
        )

    else:
        # 평가 데이터는 원본 객체 수 비율 유지
        empty_n = round(len(empty) / len(records) * target_size)
        multi_n = round(len(multi) / len(records) * target_size)
        single_n = target_size - empty_n - multi_n

        selected = sample_single(single, single_n, rng)

        if empty_n:
            selected += rng.sample(empty, empty_n)

        if multi_n:
            selected += rng.sample(multi, multi_n)

    return sorted(selected, key=lambda r: r["image"].name)


## 3. 각 split에서 1/3 추출
각 split의 실제 이미지 수에 `1/3`을 곱해 자동 계산.


In [8]:
original = {}
subset = {}

for index, split in enumerate(SPLITS):
    records = load_records(split)

    # 원본 split의 1/3
    target_size = round(len(records) * SUBSET_RATIO)

    rng = random.Random(SEED + index)
    selected = select_subset(
        records,
        split,
        target_size,
        rng,
    )

    original[split] = records
    subset[split] = selected

    print(
        f"{split:5} | "
        f"original={len(records):4} | "
        f"subset={len(selected):4} | "
        f"ratio={len(selected)/len(records):.3f}"
    )


train | original=5200 | subset=1733 | ratio=0.333
val   | original=2600 | subset= 867 | ratio=0.333
test  | original=2200 | subset= 733 | ratio=0.333


## 4. 원본과 Subset 통계 비교

확인 항목
- 이미지 수
- 객체 수
- 배경 이미지 수
- 다중 객체 수 / 비율
- bbox 면적 중앙값
- bbox 1% 미만 / 5% 미만 비율


In [9]:
def get_stats(records):
    areas = np.array([
        area
        for r in records
        for area in r["areas"]
    ])

    multi = sum(r["count"] >= 2 for r in records)

    return {
        "images": len(records),
        "objects": len(areas),
        "empty": sum(r["count"] == 0 for r in records),
        "multi": multi,
        "multi_ratio(%)": multi / len(records) * 100,
        "bbox_median(%)": np.median(areas) * 100 if len(areas) else 0,
        "bbox<1%(%)": (areas < 0.01).mean() * 100 if len(areas) else 0,
        "bbox<5%(%)": (areas < 0.05).mean() * 100 if len(areas) else 0,
    }


rows = []

for split in SPLITS:
    rows.append({
        "split": split,
        "data": "original",
        **get_stats(original[split]),
    })

    rows.append({
        "split": split,
        "data": "subset",
        **get_stats(subset[split]),
    })

pd.DataFrame(rows).round(4)


,split,data,images,objects,empty,multi,multi_ratio(%),bbox_median(%),bbox<1%(%),bbox<5%(%)
0,train,original,5200,5243,3,29,0.5577,0.0472,88.3273,94.8312
1,train,subset,1733,1776,3,29,1.6734,0.0489,87.2748,95.1577
2,val,original,2600,2621,0,13,0.5000,0.0459,88.5540,94.2388
3,val,subset,867,876,0,4,0.4614,0.0457,89.3836,94.6347
4,test,original,2200,2245,0,33,1.5000,0.0911,75.8575,92.8285
5,test,subset,733,749,0,11,1.5007,0.0941,76.3685,93.0574


## 5. bbox 크기 분포 비교

소형 드론 특성을 보기 위해 bbox 면적 5% 이하 구간을 비교.
원본과 subset 분포가 크게 달라지지 않는지 확인.


In [11]:
def bbox_stats(records):
    """bbox 크기 분포 요약."""
    areas = np.array([
        a * 100
        for r in records
        for a in r["areas"]
    ])

    return {
        "bbox 수": len(areas),
        "최솟값(%)": areas.min(),
        "25%(%)": np.percentile(areas, 25),
        "중앙값(%)": np.median(areas),
        "75%(%)": np.percentile(areas, 75),
        "최댓값(%)": areas.max(),
        "1% 미만 비율": (areas < 1).mean() * 100,
        "5% 미만 비율": (areas < 5).mean() * 100,
    }


rows = []

for split in SPLITS:
    original_stats = bbox_stats(original[split])
    subset_stats = bbox_stats(subset[split])

    rows.append({
        "split": split,
        "data": "original",
        **original_stats,
    })

    rows.append({
        "split": split,
        "data": "subset",
        **subset_stats,
    })


bbox_df = pd.DataFrame(rows)

bbox_df.round(4)

,split,data,bbox 수,최솟값(%),25%(%),중앙값(%),75%(%),최댓값(%),1% 미만 비율,5% 미만 비율
0,train,original,5243,0.0019,0.0254,0.0472,0.1221,70.1988,88.3273,94.8312
1,train,subset,1776,0.0019,0.0257,0.0489,0.1496,64.8267,87.2748,95.1577
2,val,original,2621,0.0000,0.0247,0.0459,0.1285,68.3894,88.5540,94.2388
3,val,subset,876,0.0034,0.0245,0.0457,0.1244,68.3894,89.3836,94.6347
4,test,original,2245,0.0031,0.0347,0.0911,0.9306,47.2108,75.8575,92.8285
5,test,subset,749,0.0041,0.0347,0.0941,0.9253,47.2108,76.3685,93.0574


## 6. 최종 확인

- Train: 배경·다중 객체 전부 포함 여부
- Val/Test: 다중 객체 비율이 원본과 비슷한지 확인
- 공통: bbox 중앙값, 1% 미만, 5% 미만 비율 확인
- Test subset은 이후 변경 없이 고정
